# Image Segmentation with PyTorch

Pixel-level classification for semantic and instance segmentation.

## Learning Objectives

- Understand semantic vs instance segmentation
- Use pre-trained segmentation models
- Visualize segmentation masks
- Learn about common architectures (FCN, DeepLabV3)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

import torchvision
from torchvision import datasets
from torchvision.transforms import v2
from torchvision.models.segmentation import (
    fcn_resnet50, FCN_ResNet50_Weights,
    deeplabv3_resnet50, DeepLabV3_ResNet50_Weights,
    deeplabv3_mobilenet_v3_large, DeepLabV3_MobileNet_V3_Large_Weights
)
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
)
from torchvision.utils import draw_segmentation_masks

plt.style.use('seaborn-v0_8-whitegrid')
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"torchvision version: {torchvision.__version__}")

## 1. Segmentation Types

| Type | Description | Output |
|------|-------------|--------|
| **Semantic** | Classify each pixel into a class | Class label per pixel |
| **Instance** | Separate individual objects | Per-instance masks |
| **Panoptic** | Semantic + Instance combined | Both class and instance info |

In [ ]:
# Segmentation overview
seg_info = """
=== Segmentation Architectures ===

Semantic Segmentation:
  • FCN (Fully Convolutional Network) - First modern approach
  • U-Net - Encoder-decoder with skip connections
  • DeepLabV3+ - Atrous convolutions + ASPP

Instance Segmentation:
  • Mask R-CNN - Extends Faster R-CNN with mask branch
  • YOLACT - Real-time instance segmentation

Available in torchvision:
┌────────────────────────────────────┬──────────────────┐
│ Model                              │ mIoU (COCO-val)  │
├────────────────────────────────────┼──────────────────┤
│ FCN ResNet-50                      │ 60.5             │
│ FCN ResNet-101                     │ 63.7             │
│ DeepLabV3 ResNet-50                │ 66.4             │
│ DeepLabV3 ResNet-101               │ 67.4             │
│ DeepLabV3 MobileNet V3             │ 60.3             │
│ LRASPP MobileNet V3                │ 57.9             │
└────────────────────────────────────┴──────────────────┘
"""
print(seg_info)

## 2. Semantic Segmentation with DeepLabV3

In [ ]:
# Load DeepLabV3
weights = DeepLabV3_ResNet50_Weights.DEFAULT
model = deeplabv3_resnet50(weights=weights)
model = model.to(device)
model.eval()

# Get class information
classes = weights.meta['categories']
print(f"Number of classes: {len(classes)}")
print(f"\nVOC classes: {classes}")

In [ ]:
# Preprocessing
preprocess = weights.transforms()
print("Preprocessing transforms:")
print(preprocess)

In [ ]:
# Create a test image
def create_segmentation_test_image(size=(640, 480)):
    """Create synthetic image with distinct regions."""
    w, h = size
    img = np.ones((h, w, 3), dtype=np.uint8) * 135  # Gray sky
    
    # Ground (green)
    img[h//2:, :] = [34, 139, 34]  # Forest green
    
    # "Person" shape (skin tone)
    img[100:300, 250:350] = [255, 218, 185]  # Peach
    
    # "Car" shape (red)
    img[300:380, 450:580] = [178, 34, 34]  # Firebrick
    
    return Image.fromarray(img)


# Load real sample from CIFAR and upscale
cifar = datasets.CIFAR10(root='./datasets', train=False, download=True)
sample_pil, label = cifar[100]
sample_upscaled = sample_pil.resize((640, 480), Image.Resampling.LANCZOS)

print(f"Sample class: {cifar.classes[label]}")

# Display
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(sample_upscaled)
axes[0].set_title(f'CIFAR Sample ({cifar.classes[label]})')
axes[0].axis('off')

synthetic = create_segmentation_test_image()
axes[1].imshow(synthetic)
axes[1].set_title('Synthetic Test Image')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def segment_image(model, image, preprocess, device):
    """
    Run semantic segmentation on an image.
    
    Returns:
        predictions: Raw logits (N, C, H, W)
        segmentation_map: Class indices (H, W)
    """
    # Preprocess
    img_tensor = v2.functional.to_image(image)
    batch = preprocess(img_tensor).unsqueeze(0).to(device)
    
    # Inference
    with torch.no_grad():
        output = model(batch)
    
    # Get predictions
    predictions = output['out']  # (1, num_classes, H, W)
    segmentation_map = predictions.argmax(dim=1).squeeze(0)  # (H, W)
    
    return predictions, segmentation_map.cpu()


# Run segmentation
predictions, seg_map = segment_image(model, sample_upscaled, preprocess, device)

print(f"Output shape: {predictions.shape}")
print(f"Segmentation map shape: {seg_map.shape}")
print(f"Unique classes in prediction: {torch.unique(seg_map).tolist()}")
print(f"Class names: {[classes[i] for i in torch.unique(seg_map).tolist()]}")

In [ ]:
# Create color palette for visualization
def get_color_palette(num_classes):
    """Generate distinct colors for each class."""
    palette = plt.cm.get_cmap('tab20', num_classes)
    return (palette(np.arange(num_classes))[:, :3] * 255).astype(np.uint8)


def visualize_segmentation(image, seg_map, classes, alpha=0.6):
    """
    Visualize segmentation results with color overlay.
    """
    num_classes = len(classes)
    palette = get_color_palette(num_classes)
    
    # Create colored mask
    h, w = seg_map.shape
    colored_mask = np.zeros((h, w, 3), dtype=np.uint8)
    
    for class_idx in torch.unique(seg_map):
        mask = (seg_map == class_idx).numpy()
        colored_mask[mask] = palette[class_idx]
    
    # Resize image to match segmentation
    img_resized = np.array(image.resize((w, h)))
    
    # Blend
    blended = (img_resized * (1 - alpha) + colored_mask * alpha).astype(np.uint8)
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(image)
    axes[0].set_title('Original')
    axes[0].axis('off')
    
    axes[1].imshow(colored_mask)
    axes[1].set_title('Segmentation Mask')
    axes[1].axis('off')
    
    axes[2].imshow(blended)
    axes[2].set_title('Overlay')
    axes[2].axis('off')
    
    # Add legend for detected classes
    unique_classes = torch.unique(seg_map).tolist()
    legend_elements = [
        plt.Rectangle((0,0), 1, 1, facecolor=palette[i]/255, label=classes[i])
        for i in unique_classes
    ]
    axes[1].legend(handles=legend_elements, loc='upper right', fontsize=8)
    
    plt.tight_layout()
    plt.show()


# Visualize
visualize_segmentation(sample_upscaled, seg_map, classes)

## 3. Using torchvision Drawing Utilities

In [ ]:
def visualize_with_torchvision(image, seg_map, num_classes, alpha=0.6):
    """
    Use torchvision's draw_segmentation_masks utility.
    """
    # Convert image to tensor
    img_tensor = v2.functional.to_image(image)
    
    # Resize to match segmentation
    h, w = seg_map.shape
    img_resized = v2.functional.resize(img_tensor, [h, w])
    
    # Create boolean masks for each class
    unique_classes = torch.unique(seg_map)
    masks = torch.stack([seg_map == c for c in unique_classes])
    
    # Generate colors
    colors = [tuple(c) for c in get_color_palette(num_classes)[unique_classes.numpy()]]
    
    # Draw masks
    result = draw_segmentation_masks(
        img_resized, masks, alpha=alpha, colors=colors
    )
    
    # Display
    plt.figure(figsize=(10, 8))
    plt.imshow(result.permute(1, 2, 0).numpy())
    plt.title('Segmentation with torchvision.utils.draw_segmentation_masks')
    plt.axis('off')
    plt.show()


# Visualize
visualize_with_torchvision(sample_upscaled, seg_map, len(classes))

## 4. FCN (Fully Convolutional Network)

The original semantic segmentation architecture.

In [ ]:
# Load FCN
fcn_weights = FCN_ResNet50_Weights.DEFAULT
fcn_model = fcn_resnet50(weights=fcn_weights)
fcn_model = fcn_model.to(device)
fcn_model.eval()

fcn_preprocess = fcn_weights.transforms()

# Compare FCN vs DeepLabV3
print("=== FCN vs DeepLabV3 Comparison ===")
print(f"\nFCN architecture:")
print(f"  - Uses transpose convolutions for upsampling")
print(f"  - Simpler, faster")

print(f"\nDeepLabV3 architecture:")
print(f"  - Uses Atrous (dilated) convolutions")
print(f"  - ASPP (Atrous Spatial Pyramid Pooling) for multi-scale")
print(f"  - Higher accuracy, more compute")

In [ ]:
# Run FCN segmentation
fcn_preds, fcn_seg_map = segment_image(fcn_model, sample_upscaled, fcn_preprocess, device)

# Compare outputs
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(sample_upscaled)
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(fcn_seg_map.numpy(), cmap='tab20')
axes[1].set_title('FCN Segmentation')
axes[1].axis('off')

axes[2].imshow(seg_map.numpy(), cmap='tab20')
axes[2].set_title('DeepLabV3 Segmentation')
axes[2].axis('off')

plt.suptitle('FCN vs DeepLabV3', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Instance Segmentation with Mask R-CNN

In [ ]:
# Load Mask R-CNN
mask_weights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
mask_model = maskrcnn_resnet50_fpn_v2(weights=mask_weights)
mask_model = mask_model.to(device)
mask_model.eval()

mask_preprocess = mask_weights.transforms()
coco_classes = mask_weights.meta['categories']

print(f"Mask R-CNN trained on {len(coco_classes)} COCO classes")
print(f"\nMask R-CNN outputs:")
print(f"  - boxes: Bounding boxes [x1, y1, x2, y2]")
print(f"  - labels: Class indices")
print(f"  - scores: Confidence scores")
print(f"  - masks: Binary masks for each instance")

In [ ]:
def instance_segment(model, image, preprocess, device, threshold=0.5):
    """
    Run instance segmentation with Mask R-CNN.
    
    Returns:
        Dictionary with boxes, labels, scores, masks
    """
    # Preprocess
    img_tensor = v2.functional.to_image(image)
    batch = preprocess(img_tensor).unsqueeze(0).to(device)
    
    # Inference
    with torch.no_grad():
        predictions = model(batch)
    
    # Filter by threshold
    pred = predictions[0]
    mask = pred['scores'] >= threshold
    
    return {
        'boxes': pred['boxes'][mask].cpu(),
        'labels': pred['labels'][mask].cpu(),
        'scores': pred['scores'][mask].cpu(),
        'masks': pred['masks'][mask].cpu()  # (N, 1, H, W)
    }


# Run instance segmentation
instance_results = instance_segment(mask_model, sample_upscaled, mask_preprocess, device, threshold=0.3)

print(f"Found {len(instance_results['boxes'])} instances:")
for i in range(len(instance_results['boxes'])):
    label = coco_classes[instance_results['labels'][i].item()]
    score = instance_results['scores'][i].item()
    mask_shape = instance_results['masks'][i].shape
    print(f"  {label}: {score:.1%} (mask shape: {mask_shape})")

In [ ]:
def visualize_instance_segmentation(image, results, class_labels):
    """
    Visualize instance segmentation results.
    """
    # Convert to tensor
    img_tensor = v2.functional.to_image(image)
    
    if len(results['masks']) == 0:
        print("No instances detected")
        plt.figure(figsize=(10, 8))
        plt.imshow(image)
        plt.title('No instances detected')
        plt.axis('off')
        plt.show()
        return
    
    # Resize masks to match image size
    h, w = img_tensor.shape[1:]
    masks = results['masks'].squeeze(1)  # (N, H, W)
    
    # Resize masks if needed
    if masks.shape[1:] != (h, w):
        masks = torch.nn.functional.interpolate(
            masks.unsqueeze(0), size=(h, w), mode='bilinear'
        ).squeeze(0)
    
    # Binarize masks
    binary_masks = masks > 0.5
    
    # Generate colors
    num_instances = len(binary_masks)
    colors = [tuple(c) for c in (plt.cm.Set2(np.linspace(0, 1, num_instances))[:, :3] * 255).astype(int)]
    
    # Draw masks
    result = draw_segmentation_masks(img_tensor, binary_masks, alpha=0.5, colors=colors)
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    axes[0].imshow(image)
    axes[0].set_title('Original')
    axes[0].axis('off')
    
    axes[1].imshow(result.permute(1, 2, 0).numpy())
    axes[1].set_title('Instance Segmentation')
    axes[1].axis('off')
    
    # Add legend
    legend_elements = [
        plt.Rectangle((0,0), 1, 1, facecolor=np.array(colors[i])/255, 
                      label=f"{class_labels[results['labels'][i].item()]}: {results['scores'][i]:.1%}")
        for i in range(num_instances)
    ]
    axes[1].legend(handles=legend_elements, loc='upper right', fontsize=9)
    
    plt.tight_layout()
    plt.show()


# Visualize
visualize_instance_segmentation(sample_upscaled, instance_results, coco_classes)

## 6. Lightweight Model for Mobile/Edge

In [ ]:
# Load MobileNet-based DeepLabV3
mobile_weights = DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT
mobile_model = deeplabv3_mobilenet_v3_large(weights=mobile_weights)
mobile_model = mobile_model.to(device)
mobile_model.eval()

mobile_preprocess = mobile_weights.transforms()

# Compare speed
import time

def benchmark_segmentation(model, image, preprocess, device, num_runs=5):
    """Benchmark segmentation model speed."""
    img_tensor = preprocess(v2.functional.to_image(image)).unsqueeze(0).to(device)
    
    # Warmup
    with torch.no_grad():
        for _ in range(2):
            _ = model(img_tensor)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    # Benchmark
    start = time.time()
    with torch.no_grad():
        for _ in range(num_runs):
            _ = model(img_tensor)
            if device.type == 'cuda':
                torch.cuda.synchronize()
    
    elapsed = time.time() - start
    return elapsed / num_runs * 1000  # ms per image


print("Benchmarking segmentation models...")
deeplab_time = benchmark_segmentation(model, sample_upscaled, preprocess, device)
mobile_time = benchmark_segmentation(mobile_model, sample_upscaled, mobile_preprocess, device)

print(f"\n=== Inference Speed ===")
print(f"DeepLabV3 ResNet-50:    {deeplab_time:.1f} ms/image")
print(f"DeepLabV3 MobileNet V3: {mobile_time:.1f} ms/image")
print(f"\nSpeedup: {deeplab_time/mobile_time:.1f}x faster with MobileNet")

## 7. Segmentation Metrics

In [ ]:
def compute_iou_segmentation(pred_mask, gt_mask):
    """
    Compute IoU for segmentation masks.
    
    Args:
        pred_mask: Predicted binary mask
        gt_mask: Ground truth binary mask
    
    Returns:
        IoU score
    """
    intersection = (pred_mask & gt_mask).sum()
    union = (pred_mask | gt_mask).sum()
    return (intersection / union).item() if union > 0 else 0


def compute_dice(pred_mask, gt_mask):
    """
    Compute Dice coefficient (F1 score for segmentation).
    
    Dice = 2 * |A ∩ B| / (|A| + |B|)
    """
    intersection = (pred_mask & gt_mask).sum()
    total = pred_mask.sum() + gt_mask.sum()
    return (2 * intersection / total).item() if total > 0 else 0


# Demonstrate with synthetic masks
# Create overlapping masks
h, w = 100, 100

gt_mask = torch.zeros(h, w, dtype=torch.bool)
gt_mask[20:80, 20:80] = True  # Ground truth square

pred_good = torch.zeros(h, w, dtype=torch.bool)
pred_good[25:75, 25:75] = True  # Good prediction (slightly smaller)

pred_bad = torch.zeros(h, w, dtype=torch.bool)
pred_bad[50:100, 50:100] = True  # Bad prediction (offset)

print("=== Segmentation Metrics ===")
print(f"\nGood prediction:")
print(f"  IoU:  {compute_iou_segmentation(pred_good, gt_mask):.3f}")
print(f"  Dice: {compute_dice(pred_good, gt_mask):.3f}")

print(f"\nBad prediction (offset):")
print(f"  IoU:  {compute_iou_segmentation(pred_bad, gt_mask):.3f}")
print(f"  Dice: {compute_dice(pred_bad, gt_mask):.3f}")

In [ ]:
# Visualize mask comparison
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(gt_mask.numpy(), cmap='Blues')
axes[0].set_title('Ground Truth')
axes[0].axis('off')

axes[1].imshow(pred_good.numpy(), cmap='Greens')
axes[1].set_title('Good Prediction\n(IoU=0.69, Dice=0.82)')
axes[1].axis('off')

axes[2].imshow(pred_bad.numpy(), cmap='Reds')
axes[2].set_title('Bad Prediction\n(IoU=0.16, Dice=0.27)')
axes[2].axis('off')

# Overlay for good prediction
overlay = np.zeros((h, w, 3))
overlay[gt_mask.numpy()] = [0, 0, 1]  # Blue for GT
overlay[pred_good.numpy()] = [0, 1, 0]  # Green for pred
overlap = gt_mask.numpy() & pred_good.numpy()
overlay[overlap] = [0, 1, 1]  # Cyan for overlap
axes[3].imshow(overlay)
axes[3].set_title('Overlap Visualization\n(Blue=GT, Green=Pred, Cyan=Both)')
axes[3].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Metrics overview
metrics_info = """
=== Segmentation Metrics Summary ===

┌────────────────┬─────────────────────────────────────────────────┐
│ Metric         │ Description                                     │
├────────────────┼─────────────────────────────────────────────────┤
│ IoU (Jaccard)  │ Intersection / Union                            │
│                │ Standard metric, penalizes under/over-segment   │
├────────────────┼─────────────────────────────────────────────────┤
│ Dice (F1)      │ 2 * Intersection / (|A| + |B|)                  │
│                │ Popular in medical imaging, related to IoU     │
├────────────────┼─────────────────────────────────────────────────┤
│ Pixel Accuracy │ Correct pixels / Total pixels                   │
│                │ Can be misleading with class imbalance          │
├────────────────┼─────────────────────────────────────────────────┤
│ mIoU           │ Mean IoU across all classes                     │
│                │ Standard benchmark metric                       │
└────────────────┴─────────────────────────────────────────────────┘

Relationship between IoU and Dice:
  Dice = 2 * IoU / (1 + IoU)
  IoU = Dice / (2 - Dice)
"""
print(metrics_info)

## Summary

### Segmentation Types

| Type | Use Case | Model |
|------|----------|-------|
| Semantic | Scene understanding | DeepLabV3, FCN |
| Instance | Object counting | Mask R-CNN |
| Panoptic | Comprehensive scene | PanopticFPN |

### Model Selection

| Priority | Recommended Model |
|----------|------------------|
| Accuracy | DeepLabV3 ResNet-101 |
| Speed (Mobile) | DeepLabV3 MobileNet V3 |
| Instance | Mask R-CNN ResNet-50 FPN V2 |

### Key Takeaways

1. Semantic segmentation classifies pixels, not instances
2. Instance segmentation provides per-object masks
3. DeepLabV3 uses atrous convolutions for multi-scale features
4. mIoU is the standard benchmark metric

In [ ]:
print("Notebook completed successfully!")